In [20]:
# THINK OF SOMETHING TO BE CREATIVE
# MAKE THE CODE SCALABLE 
# OR SUPER ACCURATE
state['product_catalogue']

[{'id': 'p1',
  'name': 'Wireless Headphones X200',
  'category': 'electronics',
  'price': 199.99,
  'rating': 4.5,
  'popularity': 520},
 {'id': 'p2',
  'name': 'Smartphone Z Ultra',
  'category': 'electronics',
  'price': 899.99,
  'rating': 4.7,
  'popularity': 830},
 {'id': 'p3',
  'name': 'Gaming Laptop Pro 15',
  'category': 'electronics',
  'price': 1499.99,
  'rating': 4.6,
  'popularity': 620},
 {'id': 'p4',
  'name': 'Bluetooth Speaker Mini',
  'category': 'electronics',
  'price': 49.99,
  'rating': 4.2,
  'popularity': 410},
 {'id': 'p5',
  'name': '4K Smart TV 55"',
  'category': 'electronics',
  'price': 699.99,
  'rating': 4.8,
  'popularity': 540},
 {'id': 'p6',
  'name': 'Wireless Charger Pad',
  'category': 'electronics',
  'price': 29.99,
  'rating': 4.1,
  'popularity': 390},
 {'id': 'p7',
  'name': 'Noise-Cancelling Earbuds',
  'category': 'electronics',
  'price': 129.99,
  'rating': 4.3,
  'popularity': 460},
 {'id': 'p8',
  'name': 'Smartwatch Fit S3',
  'categ

In [1]:
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from typing import TypedDict, List, Dict
from langchain_groq import ChatGroq
from pydantic import BaseModel, ValidationError, Field
from langchain.prompts import PromptTemplate
from langchain.tools import tool,Tool
from langchain.chains import ConversationChain, LLMChain
from langchain.memory import ConversationBufferMemory
from langchain.agents import initialize_agent, AgentType
import json

In [2]:
class AgentState(TypedDict):
    user_profile_path: str
    product_catalog_path: str
    unique_categories: List[str]
    product_catalogue: List[dict]
    user_profile: dict
    justification: str
    recommended_category: str
    top3_products: List[dict]
    chosen_product: dict
    history_log_file: str
    error: str 

In [3]:
groq_api_keys = ["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", "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", "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","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", "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", "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"]

In [4]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.6,
    api_key = groq_api_keys[0]
)

# give a prompt to take the user history and the unique categories and choose a likeable category
category_prompt = PromptTemplate.from_template("""
You are an intelligent shopping recommendation assistant.

You will receive:
1. The user's purchase history as a JSON list.
2. A list of unique product categories available in the catalog.

Your task:
- Analyze the user's purchase history.
- Identify categories the user has **not purchased from** before.
- If all categories have been purchased from, choose one the user has **purchased from the least**.
- Choose **one category** that the user might be most interested in exploring next.
- Generate a **catchy, persuasive justification** in natural language explaining *why* this category fits the user’s preferences or lifestyle.

Return your final answer **strictly** in the following JSON format:

{{
  "recommended_category": "<category_name>",
  "justification": "<short persuasive message>"
}}

### USER HISTORY:
{user_history}

### AVAILABLE CATEGORIES:
{available_categories}

Now respond only with the JSON object described above — no extra text.
""")
category_chain = LLMChain(llm=llm, prompt=category_prompt)



/var/folders/ly/t7r_d3hx39g_jfx68mqwwfym0000gn/T/ipykernel_93134/1047735980.py:37: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  category_chain = LLMChain(llm=llm, prompt=category_prompt)


In [5]:
#node 1
# add conditional edges
def category_extraction(state ,user_profile_path ,product_catalog_path ):
    state["user_profile_path"] = user_profile_path
    state["product_catalog_path"] = product_catalog_path
    # read user profile
    try:
        with open(state["user_profile_path"], "r") as f:
            state["user_profile"] = json.load(f)
    except FileNotFoundError:
        state["error"] = f"❌ File path: {state['user_profile_path']} not found. Please check the path."
        return state
    except json.JSONDecodeError:
        state["error"] = f"❌ Invalid JSON format in {state['user_profile_path']}."
        return state

    # read product catalogue
    try:
        with open(state["product_catalog_path"], "r") as f:
            state["product_catalogue"] = json.load(f)
            state["unique_categories"] = sorted({item["category"] for item in state["product_catalogue"]})
    except FileNotFoundError:
        state["error"] = f"❌ File path: {state['product_catalog_path']} found. Please check the path."
        return state
    except json.JSONDecodeError:
        state["error"] = f"❌ Invalid JSON format in {state['product_catalog_path']}."
        return state
    return state


In [6]:
state = AgentState()
state = category_extraction(state, "./user_profile.json", "./catalog1.json")

In [7]:
# if no error go to node 2
# node 2 

class CategoryRecommendation(BaseModel):
    recommended_category: str = None
    justification: str = None


def category_recommendation(state):
    try:
        # Run the LLM chain with provided user data
        output = category_chain.run(
            user_history=state["user_profile"]["history"],
            available_categories=state["unique_categories"]
        )

        # Parse JSON
        data = json.loads(output.strip())

        # Validate using your Pydantic model
        validated = CategoryRecommendation(**data)

        # Additional safeguard: ensure both fields are not empty or whitespace
        if not validated.recommended_category.strip() or not validated.justification.strip():
            state["error"] = "❌ One or more fields are empty or invalid in LLM output."
            return state

        # Assign validated values to state
        state["recommended_category"] = validated.recommended_category.strip()
        state["justification"] = validated.justification.strip()


        # CHECK FOR CORRECTNESS

    except json.JSONDecodeError:
        state["error"] = "❌ Invalid JSON format in LLM output."
        return state
    except ValidationError as e:
        state["error"] = f"❌ Invalid LLM output: {str(e)}"
        return state
    except Exception as e:
        state["error"] = f"❌ Unexpected error: {str(e)}"
        return state

    # If everything passes, clear error and return
    return state


In [8]:
state = category_recommendation(state)

/var/folders/ly/t7r_d3hx39g_jfx68mqwwfym0000gn/T/ipykernel_93134/4117680459.py:12: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  output = category_chain.run(


In [9]:
recommendation_prompt = PromptTemplate.from_template("""
You are a smart shopping assistant.

You are given:
- The user's purchase history: {user_history}
- A list of available products: {products}
- justification message to convince the user to explore the category : {justification}

Steps to follow:
1. Calculate the user's average spending based on their purchase history.
2. Rank the products based on how close their price is to this average and any inferred preferences of the user.
3. Choose the **top 3 closest products** to recommend.
4. Write a short, friendly **message** summarizing your recommendations for the user and mix it with the justification message.
5. Also, return a **JSON object** containing the selected products.


⚠️ Important Instructions:
- When you have completed your analysis and are ready to give your final recommendation, you must return your answer exactly in the format below.
- Once you output your final answer, stop reasoning and do not take any further actions.

Your response must be a **valid JSON object only**, with no extra text, markdown, or explanations,
Your response **must follow this exact format**:

Final Answer:
{{
  "average_price": "<computed_average_as_number>",
  "message":"<Your friendly, natural language message to the user>"
  "top_3_products": [
    {{"product_id": "<id1>", "product_name": "<name1>", "price": <price1>}},
    {{"product_id": "<id2>", "product_name": "<name2>", "price": <price2>}},
    {{"product_id": "<id3>", "product_name": "<name3>", "price": <price3>}}
  ]
}}

Now respond **strictly** following this format and stop after your final answer.
""")


In [10]:
@tool("average_calculator", return_direct=True)
def average_calculator(numbers: str) -> str:
    """
    Calculates the average of a comma-separated list of numbers.
    Example input: "120, 80, 200"
    """
    try:
        nums = [float(n.strip()) for n in numbers.split(",") if n.strip()]
        if not nums:
            return "No valid numbers provided."
        avg = sum(nums) / len(nums)
        return f"The average is {avg:.2f}"
    except Exception as e:
        return f"Error calculating average: {str(e)}"

@tool("rank_products_by_difference", return_direct=True)
def rank_products_by_difference(differences: str) -> str:
    """
    Rank products based on their absolute differences in price.

    Args:
        differences (str): A JSON string or comma-separated key-value pairs like:
                           '{"Perfume Elegant Rose": 6.20, "Electric Shaver": 3.80}'
    Returns:
        str: A sorted list of tuples (product_name, difference) in ascending order.
    """

    try:
        if isinstance(differences, str):
            differences = json.loads(differences)

        if not isinstance(differences, dict):
            return """❌ Invalid input. 
            Args:
            differences (str): A JSON string or comma-separated key-value pairs like:
                               '{"Perfume Elegant Rose": 6.20, "Electric Shaver": 3.80}'
            Do not forget that the values are the difference between the average and the product price
            """

        ranked = sorted(differences.items(), key=lambda x: x[1])

        return json.dumps(ranked, indent=2)

    except json.JSONDecodeError:
        return """❌ Invalid JSON format in input.
        Args:
        differences (str): A JSON string or comma-separated key-value pairs like:
                           '{"Perfume Elegant Rose": 6.20, "Electric Shaver": 3.80}'
        Do not forget that the values are the difference between the average and the product price
        """
    except Exception as e:
        return f"❌ Error: {str(e)}"


tools = [
    Tool(
        name="Average Calculator",
        func=average_calculator,
        description="Use this tool to calculate the average of the prices of the user's history purchased products, Give the tool a the list of prices in that format: 100.00, 200.99, ..., 50.98 "
    ),
    Tool(
        name="Product Ranking",
        func=rank_products_by_difference,
        description="Use this tool to rank the products based on the absolute difference"
    )
]


memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent_type=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True,
    handle_parsing_errors=True
)

/var/folders/ly/t7r_d3hx39g_jfx68mqwwfym0000gn/T/ipykernel_93134/220480529.py:69: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
/var/folders/ly/t7r_d3hx39g_jfx68mqwwfym0000gn/T/ipykernel_93134/220480529.py:71: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-rea

In [11]:
def product_recommender(state):
    available_products = [item for item in state["product_catalogue"] if state["recommended_category"] == item["category"]]
    
    input_prompt = recommendation_prompt.format(
    user_history=state["user_profile"]["history"],
    products=available_products,
    justification = state["justification"]
    )
    result = agent.invoke(input_prompt) 
    output = json.loads(result['output'])
    state['top3_products'] = output['top_3_products']
    # print out the 
    return result

In [ ]:
result= product_recommender(state)

In [13]:
# graph.add_conditional_edges(
#     "recommend_category_node",
#     lambda state: "error_handler" if "error" in state else "recommend_products_node"
# )
# we want to parse the output and save the message and the top 3 products 
# then ask the LLM to maybe the same agent with the extracted content
# then print out the message and take the input from the user and match the input with one of the products intelligently
# then if matched well then move to node 4 and print out a success message if not then print out a failure message 
result['output']

'{\n  "average_price": 83.79,\n  "message": "Discover a new you with our beauty range, perfectly complementing your active lifestyle and love for innovative electronics and great literature! We recommend you check out the Electric Shaver SmoothSkin, Perfume Elegant Rose 50ml, and Makeup Palette 24 Colors, which are close to your average spending and might interest you.",\n  "top_3_products": [\n    {"product_id": "p45", "product_name": "Electric Shaver SmoothSkin", "price": 79.99},\n    {"product_id": "p43", "product_name": "Perfume Elegant Rose 50ml", "price": 89.99},\n    {"product_id": "p46", "product_name": "Makeup Palette 24 Colors", "price": 49.99}\n  ]\n}'

In [19]:
d = json.loads(result['output'])
d['top_3_products']

[{'product_id': 'p45',
  'product_name': 'Electric Shaver SmoothSkin',
  'price': 79.99},
 {'product_id': 'p43',
  'product_name': 'Perfume Elegant Rose 50ml',
  'price': 89.99},
 {'product_id': 'p46',
  'product_name': 'Makeup Palette 24 Colors',
  'price': 49.99}]

In [5]:
from groq import Groq

client = Groq(api_key="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")

models = client.models.list()

for model in models.data:
    print(model.id)

playai-tts-arabic
playai-tts
qwen/qwen3-32b
meta-llama/llama-4-maverick-17b-128e-instruct
allam-2-7b
meta-llama/llama-4-scout-17b-16e-instruct
openai/gpt-oss-120b
openai/gpt-oss-20b
whisper-large-v3-turbo
moonshotai/kimi-k2-instruct
whisper-large-v3
groq/compound-mini
llama-3.3-70b-versatile
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m
llama-3.1-8b-instant
moonshotai/kimi-k2-instruct-0905
groq/compound
meta-llama/llama-guard-4-12b


In [ ]:
# Create the graph
graph = StateGraph(ShoppingState)

# Add nodes
graph.add_node("category_agent", category_agent)
graph.add_node("product_agent", product_agent)

# Define edges (category -> product -> END)
graph.add_edge("category_agent", "product_agent")
graph.add_edge("product_agent", END)

# Set the entry point
graph.set_entry_point("category_agent")

# Compile the graph
shopping_graph = graph.compile()

In [ ]:
# el haye7sal kal aaty 
# e7na hana5od el user history we nedeeha lel agent
# then el agent hayshoof random category min el categories eli mosh mawgooden fel buyers history